In [1]:
import multiprocessing
import os
import re
import csv
import random
import torch
import pandas as pd
import numpy as np
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_community.chat_models import ChatLlamaCpp

In [2]:
# Random States
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

In [3]:
# Silence llama.cpp
os.environ["LLAMA_LOG_LEVEL"] = "ERROR" 

In [4]:
def generate_notes(system_role_prompt, note_query_prompt, input_data, temperature, model, completions, filename, dataset):
  # generate and save notes

    def parse_and_clean_reports(results):
        # parse and clean reports from openai completions
        reports = []

        for result in results:
            text = result.get("output", "")
            text = text.replace('\n', ' ')
            pattern = r'\*\*\*|\s(?=\d{1,2}[\.,]{1,2}\s)' # split by *** or numbers followed by . or , 
            # Split by *** or quotes
            splits = re.split(pattern, text)
            for item in splits:
                if item:
                    cleaned = re.sub(r'^\s*\d{1,2}[\.,]{1,3}\s*', '', item)
                    cleaned = cleaned.strip()
                    # Keep only items with letters, no colon or quote
                    if cleaned and re.search(r'[a-zA-Z]', cleaned):  # keep only if contains letters
                        reports.append(cleaned)

        return reports

    def save_reports(reports, filename):
        df = pd.DataFrame(reports, columns=['report'])
        try:
            df.to_csv(filename, index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', lineterminator='\n')
            print(f"Reports saved successfully to {filename}")
        except Exception as e:
            print(f"Failed to save reports: {str(e)}")

    data_dir = f'./data/{dataset}'  # update if needed
    os.makedirs(data_dir, exist_ok=True)
    report_filepath = os.path.join(data_dir, f'{filename}.csv')

    # initialize local model
    llm = ChatLlamaCpp(
    model_path=model,
    temperature=temperature,
    n_ctx=10000,
    n_gpu_layers=8,
    n_batch=300,
    max_tokens=512,
    n_threads=max(1, multiprocessing.cpu_count() - 1),
    repeat_penalty=1.5,
    top_p=0.5,
    verbose=False,
    )

    # create prompts
    role_prompt = PromptTemplate(template=system_role_prompt['message'], input_variables=system_role_prompt['inputs'])
    note_prompt = PromptTemplate(template=note_query_prompt['message'], input_variables=note_query_prompt['inputs'])

    # create llmchains
    role_chain = LLMChain(llm=llm, prompt=role_prompt, output_key = "intermediate_output")
    note_chain = LLMChain(llm=llm, prompt=note_prompt, output_key = "output")
    # create sequentialchain
    sequential_chain = SequentialChain(
        chains=[role_chain, note_chain], input_variables = (system_role_prompt['inputs'] + note_query_prompt['inputs']), output_variables = ["output"]
    )

    # generate a number of different responses
    results = []
    for _ in range(completions):
        response = sequential_chain.invoke(input_data)
        results.append(response)

    # clean results
    results = parse_and_clean_reports(results)
    # save results
    save_reports(results, report_filepath)


In [5]:
system_role_prompt = {'message':
                      '''
                      You are a specialist in generating fictitious {domain} data for natural language processing projects.
                      You speak {language}.
                      ''' ,
                      'inputs':["domain", "language"]}

note_query_prompt = {'message':
                      '''
                      This is an example of a text similar to the one you would write: "{example_note}"

                      Other texts may include other topics. 
                      {topic_keywords}

                      Make up {number_of_reports} such texts for the {domain} domain. Return only the reports, with each report separated by "***" and nothing else. Vary the sentence structure and style.
                      ''' ,
                      'inputs':["example_note", "topic_keywords", "domain", "number_of_reports"]}

In [6]:
# Get input prompt data.
df = pd.read_csv('../promptDataPreparation/promptDataPreparation.csv')

In [7]:
completions = 2 # completions = 25
model = './models/Phi-4-mini-instruct.Q8_0.gguf'
temperature = 1.1

unique_files = []
for row in df.iterrows():
    topic_model = row[1]['topic_model']
    dataset = row[1]['dataset']
    domain = row[1]['domain']
    topic_keywords = row[1]['topic_keywords']
    example_notes = row[1]['example_notes'].split('*** SEPARATION ***')
    for index, note in enumerate(example_notes):
        # Check that the index numbers match for file saving.
        assert index % len(example_notes) == example_notes.index(note), "There was a problem!"
        input_data = {'domain': domain, 'language': 'English', 'example_note': note, 'topic_keywords': topic_keywords, 'number_of_reports': 25}
        generate_notes(system_role_prompt, note_query_prompt, input_data, temperature, model, completions, f"{index % len(example_notes)}-{topic_model}", dataset)

llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
/var/folders/12/p5c0vdcj3yx9xb69fcd0n3j80000gn/T/ipykernel_17051/3536857066.py:55: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  role_chain = LLMChain(llm=llm, prompt=role_prompt, output_key = "intermediate_output")


Reports saved successfully to ./data/yahoo/0-MATAVE.csv


llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Reports saved successfully to ./data/yahoo/1-MATAVE.csv


llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


...

Reports saved successfully to ./data/yahoo/3-MATAVE.csv


llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Reports saved successfully to ./data/yahoo/0-LDA.csv


llama_context: n_ctx_seq (10240) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


KeyboardInterrupt: 